# 03 · ¿Hay cadenas o formatos donde se rompe más?

**Objetivo:** Mostrar a revenue management y category management cómo se comporta en el anaquel la arquitectura de precios por tamaño de marcas líderes, y dónde se rompe.

**Pregunta:** ¿La ruptura depende de la cadena o del producto?

Importa porque dice a quién reclamar: si se rompe igual en todas las cadenas, la escalera viene del precio de
lista del fabricante; si se concentra en algunas, es decisión de la cadena. Grupos con 20 escalones o más.

In [1]:
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "1")
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import entorno
entorno.setup()
import profeco, tablas, pares, estadistica as E, charts as C
import escalera as S
PROC, META, CHARTS, SOCIAL = entorno.PROC, entorno.META, entorno.CHARTS, entorno.SOCIAL
CFG = entorno.config()
reg = entorno.registrar
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 90)
pts, st, cat = S.load()
REPS = 500
p = pd.read_csv(PROC / "escalones.csv")
SEL = json.loads((META / "parametros.json").read_text(encoding="utf-8"))["seleccionadas"]
bf = S.by(p, "formato", reps=REPS).sort_values("pct_roto", ascending=False)
display(bf.round(1))
mx = S.by(p, ["escalera", "formato"], min_n=10, reps=100).pivot(index="formato", columns="escalera", values="pct_roto")
display(mx.round(0))

,formato,escalones,tiendas,municipios,pct_roto,pct_plano,cambio_mediano,cambio_roto_mediano,ic90_inf,ic90_sup
2,Chedraui Selecto,23,3,3,34.8,0.0,-13.9,7.1,11.1,63.2
12,Super Chedraui,25,3,3,32.0,12.0,-8.6,7.1,10.7,56.5
1,Chedraui,176,25,25,30.7,8.5,-9.4,5.9,10.8,56.0
6,La Comer,60,8,8,26.7,5.0,-14.5,5.9,7.0,48.3
15,Walmart Express,158,19,13,19.6,1.3,-11.6,5.8,0.0,45.0
11,Sumesa,21,4,3,19.0,4.8,-13.9,4.1,0.0,42.9
0,Bodega Aurrerá,355,47,40,18.0,9.9,-13.4,3.7,1.4,38.9
5,Hipermercado Soriana,317,45,40,16.7,4.7,-15.3,2.5,3.6,34.7
14,Walmart,413,52,45,16.2,1.2,-13.9,2.4,1.1,33.3
8,Mega Soriana,163,21,20,14.7,6.7,-16.0,2.6,1.4,31.7


escalera,Chiles La Costeña en rajas,Coca-Cola,Huevo El Calvario,Huevo San Juan,Zucaritas
formato,,,,,
Bodega Aurrerá,0.0,0.0,47.0,49.0,3.0
Bodega Aurrerá Express,NaN,0.0,NaN,NaN,NaN
Chedraui,41.0,2.0,45.0,53.0,23.0
HEB,9.0,NaN,NaN,NaN,NaN
Hipermercado Soriana,13.0,0.0,34.0,50.0,2.0
La Comer,50.0,0.0,NaN,36.0,36.0
Ley,12.0,0.0,NaN,NaN,NaN
Mega Soriana,10.0,0.0,24.0,55.0,0.0
Mercado Soriana,0.0,11.0,33.0,14.0,0.0


**¿Es la cadena o la mezcla de productos?** Las cadenas no venden la misma mezcla de escaleras. Se compara cada
formato con el % roto esperado si tuviera las tasas promedio de cada escalón (mezcla igual).

In [2]:
rate = p.groupby(["escalera", "escalon"]).cambio_pct.apply(lambda v: (v > S.TOL).mean())
p["esperado"] = [rate[(a, b)] for a, b in zip(p.escalera, p.escalon)]
adj = p.groupby("formato").agg(n=("cambio_pct", "size"), observado=("cambio_pct", lambda v: (v > S.TOL).mean() * 100),
                               esperado=("esperado", lambda v: v.mean() * 100)).query("n >= 20")
adj["diferencia_pp"] = adj.observado - adj.esperado
display(adj.sort_values("diferencia_pp").round(1))
sj = p[(p.escalera == "Huevo San Juan") & (p.escalon == "12 pzas → 18 pzas")]
sjf = S.by(sj, "formato", min_n=10, reps=REPS).sort_values("pct_roto")
display(sjf.round(1))

,n,observado,esperado,diferencia_pp
formato,,,,
Superissste,114,0.0,15.7,-15.7
Mercado Soriana,74,12.2,22.9,-10.7
Soriana Super,43,11.6,17.1,-5.5
Mega Soriana,163,14.7,20.0,-5.3
Fresko,22,13.6,17.6,-4.0
Ley,40,10.0,13.9,-3.9
Hipermercado Soriana,317,16.7,19.2,-2.5
Sumesa,21,19.0,19.1,-0.1
Bodega Aurrerá,355,18.0,16.4,1.7


,formato,escalones,tiendas,municipios,pct_roto,pct_plano,cambio_mediano,cambio_roto_mediano,ic90_inf,ic90_sup
0,Bodega Aurrerá,37,37,31,94.6,5.4,3.7,3.7,88.1,100.0
2,Hipermercado Soriana,29,29,26,96.6,3.4,2.5,2.5,90.0,100.0
1,Chedraui,20,20,20,100.0,0.0,3.7,3.7,100.0,100.0
3,Mega Soriana,16,16,16,100.0,0.0,2.5,2.5,100.0,100.0
4,Walmart,38,38,33,100.0,0.0,2.4,2.4,100.0,100.0
5,Walmart Express,14,14,9,100.0,0.0,2.4,2.4,100.0,100.0


In [3]:
bf.to_csv(PROC / "rupturas_por_formato.csv", index=False, encoding="utf-8-sig")
mx.to_csv(PROC / "rupturas_formato_marca.csv", encoding="utf-8-sig")
adj.to_csv(PROC / "rupturas_formato_ajustadas.csv", encoding="utf-8-sig")
sjf.to_csv(PROC / "huevo_sj_por_formato.csv", index=False, encoding="utf-8-sig")
F = "03_cadenas_y_formatos"
reg("formato_max", bf.pct_roto.max(), fuente=F, nota=bf.iloc[0].formato); reg("formato_min", bf.pct_roto.min(), fuente=F, nota=bf.iloc[-1].formato)
reg("formatos_n", len(bf), fuente=F)
reg("ajuste_max_pp", adj.diferencia_pp.abs().max(), fuente=F, nota=adj.diferencia_pp.abs().idxmax())
reg("sj_formato_min", sjf.pct_roto.min(), fuente=F, nota=sjf.iloc[0].formato); reg("sj_formato_max", sjf.pct_roto.max(), fuente=F, nota=sjf.iloc[-1].formato)
reg("sj_formatos", len(sjf), fuente=F)
display(Markdown(
    f"**Conclusión.** El % roto va de {bf.pct_roto.min():.0f}% en {bf.iloc[-1].formato} a {bf.pct_roto.max():.0f}% en {bf.iloc[0].formato}, pero "
    f"con la misma mezcla de escalones la diferencia frente a lo esperado es de a lo más {adj.diferencia_pp.abs().max():.0f} puntos. "
    f"El escalón de 12 a 18 huevos San Juan está roto en {sjf.pct_roto.min():.0f}% a {sjf.pct_roto.max():.0f}% de las tiendas de los "
    f"{len(sjf)} formatos con datos: la ruptura viene del producto, no de la cadena."))

**Conclusión.** El % roto va de 0% en Superissste a 35% en Chedraui Selecto, pero con la misma mezcla de escalones la diferencia frente a lo esperado es de a lo más 16 puntos. El escalón de 12 a 18 huevos San Juan está roto en 95% a 100% de las tiendas de los 6 formatos con datos: la ruptura viene del producto, no de la cadena.